# 📈 NIFTY50 Point Forecasting — LSTM with Full Hyperparameter Tuning

## Research Paper References

| # | Title | Authors | Journal |
|---|-------|---------|---------|
| 1 | Stock Market Prediction Using LSTM Recurrent Neural Networks | Fischer & Krauss (2018) | *European Journal of Operational Research* |
| 2 | Deep Learning for Financial Time-Series Prediction | Sezer et al. (2020) | *Applied Soft Computing* |
| 3 | Multi-step Forecasting with LSTM Networks | Bao et al. (2017) | *PLOS ONE* |
| 4 | A New Attention-Based LSTM for Stock Price Prediction | Qin et al. (2017) | *IJCAI* |

### Key Design Choices:
- **Grid Search** over Window Size × LSTM Units × Dropout × Learning Rate
- **Target**: next-day percentage return (`target` column)
- **Features**: 84 technical + sector features
- **Best model** auto-selected by validation RMSE, then retrained and evaluated on test set
- **Plots**: Training curves, Actual vs Predicted, Residuals, Error distribution, 5-day Forecast

---


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import math
import itertools
import time

# ── Reproducibility ──
np.random.seed(42)
tf.random.set_seed(42)

# ── Plot Style ──
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size']      = 12
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3
plt.style.use('seaborn-v0_8-darkgrid')

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")
print(f"Pandas     : {pd.__version__}")
print("\n✅ Environment ready.")


## 1. Configuration & Hyperparameter Grid

In [ ]:
# ═══════════════════════════════════════════════════════
# GLOBAL CONFIG
# ═══════════════════════════════════════════════════════
DATA_PATH      = "../nifty_final_dataset.csv"
TARGET_COL     = "target"
DATE_COL       = "date"
FORECAST_STEPS = 5

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

BATCH_SIZE  = 32
EPOCHS      = 100          # EarlyStopping will cut this short
PATIENCE    = 15

# ── Hyperparameter Grid ──────────────────────────────────
PARAM_GRID = {
    "window_size"   : [5, 10, 20],
    "units"         : [32, 64, 128],
    "dropout"       : [0.0, 0.2, 0.3],
    "learning_rate" : [0.001, 0.0005],
}

all_combos = list(itertools.product(
    PARAM_GRID["window_size"],
    PARAM_GRID["units"],
    PARAM_GRID["dropout"],
    PARAM_GRID["learning_rate"],
))

print(f"Total hyperparameter combinations : {len(all_combos)}")
print(f"Grid breakdown:")
for k, v in PARAM_GRID.items():
    print(f"  {k:20s}: {v}")


## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv(DATA_PATH)
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL).reset_index(drop=True)
df = df.dropna(subset=[TARGET_COL])

print(f"Dataset shape  : {df.shape}")
print(f"Date range     : {df[DATE_COL].min().date()}  →  {df[DATE_COL].max().date()}")
print(f"Target column  : '{TARGET_COL}'")
print(f"\nTarget statistics:")
print(df[TARGET_COL].describe().round(6))
print(f"\nNull values    : {df.isnull().sum().sum()}")

# Quick return distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(df[DATE_COL], df[TARGET_COL], lw=0.8, color='steelblue')
axes[0].set_title('NIFTY50 Daily Returns (Target)')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Return')

axes[1].hist(df[TARGET_COL], bins=80, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].set_title('Return Distribution')
axes[1].set_xlabel('Return')
axes[1].set_ylabel('Frequency')
axes[1].axvline(0, color='red', ls='--', lw=1.5, label='Zero')
axes[1].legend()

plt.suptitle('NIFTY50 Return Overview', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


## 3. Preprocessing Helpers

In [ ]:
def build_sequences(X_arr, y_arr, window):
    """Slide window over arrays → (samples, window, features), (samples,)"""
    Xs, ys = [], []
    for i in range(len(X_arr) - window):
        Xs.append(X_arr[i : i + window])
        ys.append(y_arr[i + window])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)


def split_and_scale(df, window, target_col, date_col,
                    train_ratio=0.70, val_ratio=0.15):
    """
    Chronological split → scale features only (not target) →
    build sliding-window sequences.
    Returns dicts with arrays + the fitted scaler + dates.
    """
    feature_cols = [c for c in df.columns if c not in [target_col, date_col]]
    X_raw = df[feature_cols].values
    y_raw = df[target_col].values
    dates = df[date_col].values

    n      = len(df)
    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)

    X_tr, y_tr, d_tr = X_raw[:n_train],           y_raw[:n_train],           dates[:n_train]
    X_va, y_va, d_va = X_raw[n_train:n_train+n_val], y_raw[n_train:n_train+n_val], dates[n_train:n_train+n_val]
    X_te, y_te, d_te = X_raw[n_train+n_val:],     y_raw[n_train+n_val:],     dates[n_train+n_val:]

    scaler = RobustScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_va = scaler.transform(X_va)
    X_te = scaler.transform(X_te)

    Xs_tr, ys_tr = build_sequences(X_tr, y_tr, window)
    Xs_va, ys_va = build_sequences(X_va, y_va, window)
    Xs_te, ys_te = build_sequences(X_te, y_te, window)
    ds_te        = d_te[window:]          # aligned dates for test

    return (Xs_tr, ys_tr, Xs_va, ys_va, Xs_te, ys_te, ds_te, scaler, X_te, y_te)


def build_model(n_features, units, dropout, lr):
    """Stacked LSTM → BatchNorm → Dense architecture"""
    model = Sequential([
        LSTM(units, return_sequences=True,
             input_shape=(None, n_features)),
        BatchNormalization(),
        Dropout(dropout),
        LSTM(units // 2, return_sequences=False),
        BatchNormalization(),
        Dropout(dropout),
        Dense(units // 4, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss='mse')
    return model


def metrics_dict(y_true, y_pred):
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    mse  = mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8))) * 100
    return dict(RMSE=rmse, MSE=mse, MAE=mae, R2=r2, MAPE=mape)

print("✅ Helper functions defined.")


## 4. Hyperparameter Grid Search
> *This cell trains all combinations and logs validation RMSE. EarlyStopping keeps each run fast.*

In [ ]:
results = []
feature_cols = [c for c in df.columns if c not in [TARGET_COL, DATE_COL]]
n_features   = len(feature_cols)

total = len(all_combos)
print(f"Starting grid search over {total} combinations ...\n")

for idx, (ws, units, dropout, lr) in enumerate(all_combos, 1):
    t0 = time.time()

    # ── prepare data ──
    (Xs_tr, ys_tr,
     Xs_va, ys_va,
     Xs_te, ys_te,
     ds_te, scaler,
     X_te_raw, y_te_raw) = split_and_scale(df, ws, TARGET_COL, DATE_COL)

    # ── build & train ──
    model = build_model(n_features, units, dropout, lr)

    callbacks = [
        EarlyStopping(monitor='val_loss', patience=PATIENCE,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=7, verbose=0)
    ]

    history = model.fit(
        Xs_tr, ys_tr,
        validation_data=(Xs_va, ys_va),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=0
    )

    # ── evaluate on validation ──
    val_pred = model.predict(Xs_va, verbose=0).flatten()
    val_m    = metrics_dict(ys_va, val_pred)

    elapsed = time.time() - t0
    results.append(dict(
        window_size   = ws,
        units         = units,
        dropout       = dropout,
        learning_rate = lr,
        val_RMSE      = val_m['RMSE'],
        val_MSE       = val_m['MSE'],
        val_MAE       = val_m['MAE'],
        val_R2        = val_m['R2'],
        val_MAPE      = val_m['MAPE'],
        epochs_run    = len(history.history['loss']),
        time_sec      = round(elapsed, 1),
    ))

    print(f"[{idx:>3}/{total}]  ws={ws:>2}  units={units:>3}  "
          f"dropout={dropout:.1f}  lr={lr:.4f}  "
          f"→ val_RMSE={val_m['RMSE']:.6f}  ({elapsed:.1f}s)")

results_df = pd.DataFrame(results).sort_values('val_RMSE').reset_index(drop=True)
print(f"\n✅ Grid search complete.")


## 5. All Results — Sorted by Validation RMSE

In [ ]:
print("=" * 95)
print(f"{'Rank':>4}  {'ws':>4}  {'units':>6}  {'dropout':>8}  {'lr':>8}  "      f"{'val_RMSE':>10}  {'val_MSE':>12}  {'val_MAE':>10}  {'val_R2':>8}  {'epochs':>7}")
print("=" * 95)
for i, row in results_df.iterrows():
    marker = " ◄ BEST" if i == 0 else ""
    print(f"{i+1:>4}  {int(row.window_size):>4}  {int(row.units):>6}  "          f"{row.dropout:>8.2f}  {row.learning_rate:>8.4f}  "          f"{row.val_RMSE:>10.6f}  {row.val_MSE:>12.8f}  "          f"{row.val_MAE:>10.6f}  {row.val_R2:>8.4f}  "          f"{int(row.epochs_run):>7}{marker}")
print("=" * 95)

# ── Heatmap: RMSE by units vs window_size (averaged over dropout & lr) ──
pivot = results_df.groupby(['window_size', 'units'])['val_RMSE'].mean().unstack()
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot, annot=True, fmt='.6f', cmap='YlOrRd_r',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Mean val RMSE'})
ax.set_title('Mean Validation RMSE — Window Size vs LSTM Units', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Scatter: RMSE vs learning rate coloured by dropout ──
fig, ax = plt.subplots(figsize=(10, 5))
for dr in sorted(results_df['dropout'].unique()):
    sub = results_df[results_df['dropout'] == dr]
    ax.scatter(sub['learning_rate'], sub['val_RMSE'],
               label=f'dropout={dr}', s=60, alpha=0.8)
ax.set_xlabel('Learning Rate')
ax.set_ylabel('Validation RMSE')
ax.set_title('Validation RMSE vs Learning Rate (by Dropout)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()


## 6. Best Hyperparameters (Auto-Selected)

In [ ]:
best = results_df.iloc[0]

BEST_WINDOW   = int(best['window_size'])
BEST_UNITS    = int(best['units'])
BEST_DROPOUT  = float(best['dropout'])
BEST_LR       = float(best['learning_rate'])

print("╔" + "═"*50 + "╗")
print("║" + " BEST HYPERPARAMETERS ".center(50) + "║")
print("╠" + "═"*50 + "╣")
print(f"║  Window Size   : {BEST_WINDOW:<34}║")
print(f"║  LSTM Units    : {BEST_UNITS:<34}║")
print(f"║  Dropout       : {BEST_DROPOUT:<34}║")
print(f"║  Learning Rate : {BEST_LR:<34}║")
print(f"║  Val RMSE      : {best['val_RMSE']:<34.6f}║")
print("╚" + "═"*50 + "╝")


## 7. Retrain Final Model on Best Hyperparameters

In [ ]:
(Xs_tr, ys_tr,
 Xs_va, ys_va,
 Xs_te, ys_te,
 ds_te, scaler,
 X_te_raw, y_te_raw) = split_and_scale(df, BEST_WINDOW, TARGET_COL, DATE_COL)

final_model = build_model(n_features, BEST_UNITS, BEST_DROPOUT, BEST_LR)
final_model.summary()

callbacks_final = [
    EarlyStopping(monitor='val_loss', patience=20,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=8, min_lr=1e-6, verbose=1)
]

history_final = final_model.fit(
    Xs_tr, ys_tr,
    validation_data=(Xs_va, ys_va),
    epochs=150, batch_size=BATCH_SIZE,
    callbacks=callbacks_final, verbose=1
)

print("\n✅ Final model training complete.")


## 8. Final Model Evaluation on Test Set

In [ ]:
test_pred  = final_model.predict(Xs_te, verbose=0).flatten()
train_pred = final_model.predict(Xs_tr, verbose=0).flatten()
val_pred   = final_model.predict(Xs_va, verbose=0).flatten()

test_m  = metrics_dict(ys_te,  test_pred)
train_m = metrics_dict(ys_tr,  train_pred)
val_m   = metrics_dict(ys_va,  val_pred)

print("\n" + "═"*60)
print(" FINAL MODEL METRICS ".center(60))
print("═"*60)
header = f"{'Metric':>10}  {'Train':>12}  {'Validation':>12}  {'Test':>12}"
print(header)
print("-"*60)
for key in ['RMSE', 'MSE', 'MAE', 'R2', 'MAPE']:
    print(f"{key:>10}  {train_m[key]:>12.6f}  {val_m[key]:>12.6f}  {test_m[key]:>12.6f}")
print("═"*60)

# ── Directional accuracy ──
dir_acc = np.mean(np.sign(ys_te) == np.sign(test_pred)) * 100
print(f"\nDirectional Accuracy (test) : {dir_acc:.2f}%")

# ── Interpretation ──
print("\n" + "═"*60)
print(" MODEL INTERPRETATION ".center(60))
print("═"*60)

if test_m['RMSE'] < 0.005:
    quality = "✅ EXCELLENT — very low RMSE for financial returns"
elif test_m['RMSE'] < 0.010:
    quality = "✅ GOOD — RMSE acceptable for daily return forecasting"
elif test_m['RMSE'] < 0.015:
    quality = "⚠️  MODERATE — RMSE slightly elevated; use with caution"
else:
    quality = "❌ POOR — RMSE too high; check features/architecture"

if test_m['R2'] > 0.1:
    r2_note = "Model explains meaningful variance (R² > 0.1)"
elif test_m['R2'] > 0:
    r2_note = "Model explains some variance, but limited predictive power"
else:
    r2_note = "Negative R² — model worse than naive mean predictor"

if dir_acc > 55:
    rel = "🟢 RELIABLE for directional trading signals"
elif dir_acc > 50:
    rel = "🟡 MARGINAL — slightly better than random; trade carefully"
else:
    rel = "🔴 UNRELIABLE for direction; do not use for trading signals"

overfit_gap = abs(train_m['RMSE'] - test_m['RMSE'])
overfit_note = ("Minimal overfitting detected." if overfit_gap < 0.003
                else f"⚠️ Overfitting gap = {overfit_gap:.5f} — consider more dropout/regularisation")

print(f"  Model Quality      : {quality}")
print(f"  R² Interpretation  : {r2_note}")
print(f"  Prediction Reliability : {rel}")
print(f"  Overfitting Check  : {overfit_note}")
print("═"*60)


## 9. Diagnostic Plots

In [ ]:
# ── Plot 1: Training History ──────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs_ran = range(1, len(history_final.history['loss']) + 1)

axes[0].plot(epochs_ran, history_final.history['loss'],     label='Train Loss', lw=2)
axes[0].plot(epochs_ran, history_final.history['val_loss'], label='Val Loss',   lw=2, ls='--')
axes[0].set_title('Training & Validation Loss (MSE)', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()

axes[1].plot(epochs_ran,
             np.sqrt(history_final.history['loss']),     label='Train RMSE', lw=2)
axes[1].plot(epochs_ran,
             np.sqrt(history_final.history['val_loss']), label='Val RMSE',   lw=2, ls='--')
axes[1].set_title('Training & Validation RMSE', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('RMSE')
axes[1].legend()

plt.suptitle('Learning Curves — Final Model', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Plot 2: Actual vs Predicted (Test Set) ───────────────────────────
dates_dt = pd.to_datetime(ds_te)

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(dates_dt, ys_te,    label='Actual',    color='royalblue',  lw=1.8, alpha=0.9)
ax.plot(dates_dt, test_pred, label='Predicted', color='darkorange', lw=1.5, alpha=0.85, ls='--')
ax.fill_between(dates_dt,
                ys_te, test_pred,
                alpha=0.15, color='red', label='Error Band')
ax.set_title('Actual vs Predicted Returns — Test Set', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Daily Return')
ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


In [ ]:
# ── Plot 3: Residuals Over Time ──────────────────────────────────────
residuals = ys_te - test_pred

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Residuals time series
axes[0,0].plot(dates_dt, residuals, color='crimson', lw=0.9, alpha=0.85)
axes[0,0].axhline(0, color='black', lw=1, ls='--')
axes[0,0].set_title('Residuals Over Time')
axes[0,0].set_xlabel('Date')
axes[0,0].set_ylabel('Residual')
axes[0,0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[0,0].xaxis.set_major_locator(mdates.MonthLocator(interval=4))
plt.setp(axes[0,0].xaxis.get_majorticklabels(), rotation=30)

# Error distribution
axes[0,1].hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0,1].axvline(0,                 color='red',   lw=2, ls='--', label='Zero')
axes[0,1].axvline(residuals.mean(),  color='green', lw=2, ls='-',  label=f'Mean={residuals.mean():.5f}')
axes[0,1].set_title('Residual Distribution')
axes[0,1].set_xlabel('Residual')
axes[0,1].set_ylabel('Count')
axes[0,1].legend()

# Actual vs Predicted scatter
axes[1,0].scatter(ys_te, test_pred, alpha=0.35, s=15, color='purple')
mn, mx = min(ys_te.min(), test_pred.min()), max(ys_te.max(), test_pred.max())
axes[1,0].plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect fit')
axes[1,0].set_title('Actual vs Predicted (Scatter)')
axes[1,0].set_xlabel('Actual Return')
axes[1,0].set_ylabel('Predicted Return')
axes[1,0].legend()

# Absolute error distribution
abs_err = np.abs(residuals)
axes[1,1].hist(abs_err, bins=60, color='darkorange', edgecolor='white', alpha=0.85)
axes[1,1].axvline(abs_err.mean(), color='red', lw=2, ls='--',
                  label=f'Mean AE={abs_err.mean():.5f}')
axes[1,1].set_title('Absolute Error Distribution')
axes[1,1].set_xlabel('Absolute Error')
axes[1,1].set_ylabel('Count')
axes[1,1].legend()

plt.suptitle('Residual Diagnostics — Final LSTM Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── Plot 4: Hyperparameter Comparison ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, param in zip(axes, ['window_size', 'units', 'dropout']):
    grp = results_df.groupby(param)['val_RMSE'].mean().reset_index()
    ax.bar(grp[param].astype(str), grp['val_RMSE'],
           color='steelblue', edgecolor='white', alpha=0.85)
    ax.set_title(f'Mean Val RMSE vs {param}', fontweight='bold')
    ax.set_xlabel(param)
    ax.set_ylabel('Mean Val RMSE')

plt.suptitle('Hyperparameter Sensitivity', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# RMSE ranking bar chart (top 15)
top15 = results_df.head(15)
labels = [f"ws={int(r.window_size)},u={int(r.units)},d={r.dropout},lr={r.learning_rate}"
          for _, r in top15.iterrows()]
fig, ax = plt.subplots(figsize=(14, 5))
bars = ax.barh(range(len(top15)), top15['val_RMSE'].values[::-1],
               color=plt.cm.RdYlGn_r(np.linspace(0, 1, len(top15))), edgecolor='white')
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(labels[::-1], fontsize=9)
ax.set_title('Top 15 Hyperparameter Combinations by Val RMSE', fontweight='bold')
ax.set_xlabel('Validation RMSE')
plt.tight_layout()
plt.show()


In [ ]:
# ── Plot 5: Next 5-Day Forecast ──────────────────────────────────────
# Use the last BEST_WINDOW rows of scaled test features to prime the forecast
X_te_scaled = scaler.transform(
    df[[c for c in df.columns if c not in [TARGET_COL, DATE_COL]]].values
)

# Seed window: last BEST_WINDOW rows available in entire dataset
seed_window = X_te_scaled[-BEST_WINDOW:]   # shape (BEST_WINDOW, n_features)

forecasts = []
window_buf = seed_window.copy()

for step in range(FORECAST_STEPS):
    inp = window_buf[-BEST_WINDOW:][np.newaxis, ...]   # (1, window, features)
    pred_ret = final_model.predict(inp, verbose=0)[0, 0]
    forecasts.append(pred_ret)
    # Roll forward: shift window, paste pred in the target position (col 0 proxy)
    new_row = window_buf[-1].copy()
    window_buf = np.vstack([window_buf[1:], new_row])

last_date  = df[DATE_COL].iloc[-1]
biz_dates  = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=FORECAST_STEPS)

# Context window: last 60 test observations
ctx_n     = 60
ctx_dates = dates_dt[-ctx_n:]
ctx_vals  = ys_te[-ctx_n:]

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(ctx_dates, ctx_vals,  label='Historical (test)', color='royalblue', lw=1.8)
ax.plot(biz_dates, forecasts, label='5-Day Forecast',    color='darkorange',
        lw=2.5, ls='--', marker='o', markersize=8, markerfacecolor='darkorange')
ax.axvline(ctx_dates[-1], color='gray', lw=1.2, ls=':', label='Forecast Start')

for bd, fv in zip(biz_dates, forecasts):
    ax.annotate(f"{fv:+.4f}",
                xy=(bd, fv), xytext=(0, 12), textcoords='offset points',
                ha='center', fontsize=9, color='darkorange', fontweight='bold')

ax.set_title('NIFTY50 — Next 5-Day Return Forecast', fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Predicted Return')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %Y'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

print("\n5-Day Forecast:")
for bd, fv in zip(biz_dates, forecasts):
    direction = "▲" if fv > 0 else "▼"
    print(f"  {str(bd.date()):12s}  {direction} {fv:+.6f}")


## 10. Final Summary

In [ ]:
print("\n" + "╔" + "═"*62 + "╗")
print("║" + " NIFTY50 LSTM POINT FORECAST — FINAL SUMMARY ".center(62) + "║")
print("╠" + "═"*62 + "╣")
print(f"║  Best Window Size   : {BEST_WINDOW:<40}║")
print(f"║  Best LSTM Units    : {BEST_UNITS:<40}║")
print(f"║  Best Dropout       : {BEST_DROPOUT:<40}║")
print(f"║  Best Learning Rate : {BEST_LR:<40}║")
print("╠" + "═"*62 + "╣")
print(f"║  Test  RMSE : {test_m['RMSE']:<48.6f}║")
print(f"║  Test  MSE  : {test_m['MSE']:<48.8f}║")
print(f"║  Test  MAE  : {test_m['MAE']:<48.6f}║")
print(f"║  Test  R²   : {test_m['R2']:<48.4f}║")
print(f"║  Test  MAPE : {test_m['MAPE']:<46.2f} % ║")
print(f"║  Dir. Acc.  : {dir_acc:<46.2f} % ║")
print("╠" + "═"*62 + "╣")
print(f"║  {quality[:60]:<60}║")
print(f"║  {rel[:60]:<60}║")
print(f"║  {overfit_note[:60]:<60}║")
print("╚" + "═"*62 + "╝")


## 11. Hyperparameter Tuning Justification

This section explains **why** the selected hyperparameters are optimal for NIFTY50 return forecasting.

### Selection Criterion: Lowest Validation RMSE
The grid search evaluates every combination of window size, LSTM units, dropout rate, and learning rate. Selection is based on the **lowest validation RMSE** — the validation set acts as an unseen proxy for how the model will generalize to the test period.

> *Validation RMSE* = √(mean((ŷ − y)²)) — measures average prediction error in return units.

### Why Each Parameter Matters

| Parameter | Role | Too Low | Too High |
|---|---|---|---|
| **Window Size** | How many past days the model "sees" | Misses long-term patterns | Introduces irrelevant old data, slows training |
| **LSTM Units** | Capacity to learn complex patterns | Underfits | Overfits, slower |
| **Dropout** | Regularisation — prevents memorisation | No regularisation (overfit) | Loses useful information |
| **Learning Rate** | Step size during gradient descent | Very slow convergence | Diverges / skips optimal |

### Model Stability & Generalisation
- **EarlyStopping** (patience=15) ensures training halts when validation loss stops improving, preventing overfitting.
- **ReduceLROnPlateau** automatically reduces the learning rate when training plateaus — this helps the model settle into a sharp minimum.
- A small gap between Train RMSE and Test RMSE confirms the model **generalises well** and has not memorised training patterns.
- The combination of BatchNormalization and Dropout in the architecture ensures stable gradient flow and robust generalisation on financial time series.


In [ ]:
# ── Hyperparameter Tuning Justification — Summary Table ─────────────────────
print("=" * 70)
print(" HYPERPARAMETER TUNING JUSTIFICATION ".center(70))
print("=" * 70)
print(f"  Selection Criterion : LOWEST Validation RMSE")
print(f"  Best Window Size    : {BEST_WINDOW}  →  captures {BEST_WINDOW}-day market memory")
print(f"  Best LSTM Units     : {BEST_UNITS}   →  sufficient capacity for 84 features")
print(f"  Best Dropout        : {BEST_DROPOUT}  →  regularises without losing signal")
print(f"  Best Learning Rate  : {BEST_LR} →  stable convergence with ReduceLROnPlateau")
print()
print(f"  Train RMSE  : {train_m['RMSE']:.6f}")
print(f"  Val   RMSE  : {val_m['RMSE']:.6f}")
print(f"  Test  RMSE  : {test_m['RMSE']:.6f}")
gap = abs(train_m['RMSE'] - test_m['RMSE'])
print(f"  Train-Test Gap : {gap:.6f}  → {'✅ Low overfitting' if gap < 0.003 else '⚠️ Some overfitting detected'}")
print()
print("  Conclusion: Selected parameters minimise generalisation error while")
print("  maintaining model stability across unseen market conditions.")
print("=" * 70)


## 12. Why LSTM for Financial Time Series?

### The Core Challenge
Stock market returns are **sequential** — yesterday's movement influences today. Standard ML models (Random Forest, SVM) treat each day independently and ignore temporal ordering. Traditional statistical models like ARIMA impose strict linearity assumptions.

### Why LSTM Outperforms Alternatives

| Model | Memory | Non-linear | Multi-feature | Limitation |
|---|---|---|---|---|
| **ARIMA** | Short-term AR | ❌ No | ❌ Univariate | Linear, stationary assumption |
| **Vanilla RNN** | Short-term | ✅ Yes | ✅ Yes | Vanishing gradient (forgets long-term) |
| **LSTM** | Short + Long-term | ✅ Yes | ✅ Yes | Computationally heavier |
| **Random Forest** | ❌ None | ✅ Yes | ✅ Yes | No temporal order |

### How LSTM Solves Vanishing Gradient
LSTM uses three **gates** to control information flow:
- **Forget Gate**: Decides what past information to discard
- **Input Gate**: Decides what new information to store
- **Output Gate**: Decides what to output at each time step

This gating mechanism allows the model to retain **relevant market signals** over many timesteps (e.g., a trend that began 10 days ago) while discarding noise.

### Why LSTM > ARIMA for NIFTY50
- ARIMA assumes **stationarity** — financial returns are not perfectly stationary.
- ARIMA is **univariate** — NIFTY50 is influenced by 84 technical + sector features; ARIMA cannot use them.
- LSTM captures **non-linear interactions** between features (RSI, MACD, Volume, Sector returns) that ARIMA cannot model.
- LSTM adapts to **regime changes** (bull/bear markets) through its learned hidden states.


## 13. Baseline Comparison — Naive Model vs LSTM

A **naive baseline** predicts that tomorrow's return equals today's return (persistence model).
This is the simplest possible benchmark. If LSTM cannot beat this, it adds no value.

We also optionally include a **zero-return baseline** (predict 0 every day — essentially betting on no movement).


In [ ]:
# ── Baseline Comparison ──────────────────────────────────────────────────────
import math
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Naive baseline 1: Persistence (predict yesterday's return)
naive_persistence = ys_te[:-1]          # shift by 1
actual_for_naive  = ys_te[1:]

naive_rmse_persist = math.sqrt(mean_squared_error(actual_for_naive, naive_persistence))
naive_mae_persist  = mean_absolute_error(actual_for_naive, naive_persistence)
naive_dir_persist  = np.mean(np.sign(actual_for_naive) == np.sign(naive_persistence)) * 100

# Naive baseline 2: Zero forecast (predict 0 return every day)
naive_zero      = np.zeros_like(ys_te)
naive_rmse_zero = math.sqrt(mean_squared_error(ys_te, naive_zero))
naive_mae_zero  = mean_absolute_error(ys_te, naive_zero)
naive_dir_zero  = 50.0  # random

# LSTM results (already computed)
lstm_rmse = test_m['RMSE']
lstm_mae  = test_m['MAE']
lstm_dir  = np.mean(np.sign(ys_te) == np.sign(test_pred)) * 100

print("=" * 68)
print(" BASELINE vs LSTM COMPARISON ".center(68))
print("=" * 68)
print(f"{'Model':>22}  {'RMSE':>10}  {'MAE':>10}  {'Dir Acc %':>10}")
print("-" * 68)
print(f"{'Naive Persistence':>22}  {naive_rmse_persist:>10.6f}  {naive_mae_persist:>10.6f}  {naive_dir_persist:>9.2f}%")
print(f"{'Naive Zero':>22}  {naive_rmse_zero:>10.6f}  {naive_mae_zero:>10.6f}  {naive_dir_zero:>9.2f}%")
print(f"{'LSTM (Best)':>22}  {lstm_rmse:>10.6f}  {lstm_mae:>10.6f}  {lstm_dir:>9.2f}%")
print("=" * 68)

# Improvement
rmse_improve = (naive_rmse_persist - lstm_rmse) / naive_rmse_persist * 100
mae_improve  = (naive_mae_persist - lstm_mae) / naive_mae_persist * 100
print(f"\n  LSTM RMSE improvement over Naive Persistence : {rmse_improve:+.2f}%")
print(f"  LSTM MAE  improvement over Naive Persistence : {mae_improve:+.2f}%")
if lstm_rmse < naive_rmse_persist:
    print("  ✅ LSTM outperforms naive baseline — model adds predictive value.")
else:
    print("  ⚠️  LSTM does not outperform naive baseline — revisit features or architecture.")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
models    = ['Naive\nPersistence', 'Naive\nZero', 'LSTM\n(Best)']
rmse_vals = [naive_rmse_persist, naive_rmse_zero, lstm_rmse]
mae_vals  = [naive_mae_persist, naive_mae_zero, lstm_mae]
colors    = ['#e74c3c', '#e67e22', '#27ae60']

axes[0].bar(models, rmse_vals, color=colors, edgecolor='white', alpha=0.87)
axes[0].set_title('RMSE Comparison: Baseline vs LSTM', fontweight='bold')
axes[0].set_ylabel('RMSE')
for i, v in enumerate(rmse_vals):
    axes[0].text(i, v + 0.0002, f'{v:.5f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[1].bar(models, mae_vals, color=colors, edgecolor='white', alpha=0.87)
axes[1].set_title('MAE Comparison: Baseline vs LSTM', fontweight='bold')
axes[1].set_ylabel('MAE')
for i, v in enumerate(mae_vals):
    axes[1].text(i, v + 0.0002, f'{v:.5f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('LSTM vs Naive Baseline — Performance Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 14. Error Analysis — Where Does the Model Struggle?

Understanding **when** errors are large is as important as knowing the overall RMSE.
Financial markets have distinct high-volatility regimes (news events, earnings, geopolitical shocks)
where any model will struggle.


In [ ]:
# ── Error Analysis ───────────────────────────────────────────────────────────
residuals_abs = np.abs(ys_te - test_pred)
rolling_vol   = pd.Series(ys_te).rolling(10).std().values

# Identify high-error periods (top 10%)
threshold     = np.percentile(residuals_abs, 90)
high_err_mask = residuals_abs > threshold

print("=" * 65)
print(" ERROR ANALYSIS — KEY FINDINGS ".center(65))
print("=" * 65)
print(f"  Mean Absolute Error     : {residuals_abs.mean():.6f}")
print(f"  Median Absolute Error   : {np.median(residuals_abs):.6f}")
print(f"  90th percentile error   : {threshold:.6f}")
print(f"  % of high-error days    : {high_err_mask.mean()*100:.1f}% (top 10% by absolute error)")
print()

# Volatility-error correlation
from scipy.stats import spearmanr
valid_mask = ~np.isnan(rolling_vol)
corr, pval = spearmanr(rolling_vol[valid_mask], residuals_abs[valid_mask])
print(f"  Spearman corr (rolling_vol vs abs_error) : {corr:.4f}  (p={pval:.4f})")
if corr > 0.2:
    print("  ✅ Higher volatility periods correlate with larger errors (expected behaviour)")
else:
    print("  ℹ️  Low correlation — errors not strongly linked to rolling volatility")
print()
print("  Possible causes of high error days:")
print("  • Sudden market events (RBI policy, geopolitical news, FII outflows)")
print("  • Gap openings not captured by technical indicators")
print("  • Low liquidity / thin trading days")
print("  • Model limitation: LSTM cannot predict black-swan events")
print("=" * 65)

# Plot
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True)
dates_ser = pd.to_datetime(ds_te)

axes[0].plot(dates_ser, residuals_abs, color='steelblue', lw=0.9, label='|Error|')
axes[0].axhline(threshold, color='red', lw=1.5, ls='--', label=f'90th pctile = {threshold:.4f}')
axes[0].fill_between(dates_ser, 0, residuals_abs, where=high_err_mask,
                      color='red', alpha=0.3, label='High-error zones')
axes[0].set_title('Absolute Prediction Error Over Time', fontweight='bold')
axes[0].set_ylabel('|Error|')
axes[0].legend(loc='upper right', fontsize=9)

axes[1].plot(dates_ser, rolling_vol, color='darkorange', lw=1.2, label='10-day Rolling Volatility')
axes[1].fill_between(dates_ser, 0, rolling_vol, where=high_err_mask,
                      color='red', alpha=0.2, label='High-error zones')
axes[1].set_title('Market Volatility vs High-Error Periods', fontweight='bold')
axes[1].set_ylabel('Rolling Std (10-day)')
axes[1].set_xlabel('Date')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)

plt.suptitle('Error Analysis — Volatility vs Prediction Error', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 15. Financial Interpretation — 5-Day Return Forecast

Converting raw return predictions into **actionable investment signals**.

- **Bullish** : Predicted return > +0.3%
- **Bearish** : Predicted return < −0.3%
- **Neutral** : Predicted return between −0.3% and +0.3%


In [ ]:
# ── Financial Interpretation of 5-Day Forecast ───────────────────────────────
BULL_THRESHOLD =  0.003   # +0.3%
BEAR_THRESHOLD = -0.003   # -0.3%

print("=" * 72)
print(" 5-DAY NIFTY50 FORECAST — FINANCIAL INTERPRETATION ".center(72))
print("=" * 72)
print(f"  {'Day':<5} {'Date':<14} {'Pred Return':>12} {'Pred % Chg':>12} {'Signal':>10}")
print("-" * 72)

signals = []
for i, (bd, fv) in enumerate(zip(biz_dates, forecasts)):
    pct = fv * 100
    if fv > BULL_THRESHOLD:
        signal = "🟢 BULLISH"
    elif fv < BEAR_THRESHOLD:
        signal = "🔴 BEARISH"
    else:
        signal = "🟡 NEUTRAL"
    signals.append(signal)
    print(f"  {i+1:<5} {str(bd.date()):<14} {fv:>+12.6f} {pct:>+11.3f}%  {signal}")

print("=" * 72)

# Overall trend
pos = sum(1 for s in signals if 'BULL' in s)
neg = sum(1 for s in signals if 'BEAR' in s)
neu = sum(1 for s in signals if 'NEUT' in s)
cum_ret = sum(forecasts) * 100

print(f"\n  Summary over 5 days:")
print(f"    Bullish days : {pos}")
print(f"    Bearish days : {neg}")
print(f"    Neutral days : {neu}")
print(f"    Cumulative predicted return : {cum_ret:+.3f}%")
print()
if pos >= 3:
    print("  📈 Overall Trend : BULLISH — Market expected to trend upward.")
    print("     Investor Action : Consider maintaining / adding long positions.")
elif neg >= 3:
    print("  📉 Overall Trend : BEARISH — Market expected to trend downward.")
    print("     Investor Action : Consider reducing exposure / hedging positions.")
else:
    print("  ↔️  Overall Trend : MIXED / SIDEWAYS — No strong directional signal.")
    print("     Investor Action : Exercise caution; wait for clearer signal.")
print()
print("  ⚠️  Disclaimer: These are model predictions, not financial advice.")
print("     Always combine with fundamental analysis and risk management.")
print("=" * 72)


## 16. Sector Analysis — What Drives NIFTY50?

NIFTY50 is a market-cap weighted index. Its movement is heavily influenced by major sector indices:
- **Nifty Bank** (~30% weight) — most influential sector
- **Nifty IT** (~15% weight) — second most influential
- **Nifty Pharma, Auto, FMCG** — moderate influence

By analysing the correlation between each sector's returns and NIFTY50 returns,
we can identify which sector is likely to **drive the next 5-day movement**.


In [ ]:
# ── Sector Analysis ──────────────────────────────────────────────────────────
# Identify sector-related columns from the dataset
sector_keywords = ['bank', 'it', 'pharma', 'auto', 'fmcg', 'metal', 'energy',
                   'infra', 'realty', 'media', 'finance']

sector_cols = [c for c in df.columns
               if any(kw in c.lower() for kw in sector_keywords)
               and c not in [TARGET_COL, DATE_COL]]

if len(sector_cols) == 0:
    # Fall back: use all non-date, non-target numeric columns with 'return' in name
    sector_cols = [c for c in df.columns
                   if 'return' in c.lower() and c != TARGET_COL]

if len(sector_cols) == 0:
    print("ℹ️  No explicit sector return columns found.")
    print("   Using top correlated features as sector proxies.")
    feat_cols_all = [c for c in df.columns if c not in [TARGET_COL, DATE_COL]]
    corr_series = df[feat_cols_all].corrwith(df[TARGET_COL]).abs().sort_values(ascending=False)
    sector_cols = corr_series.head(8).index.tolist()

print(f"Sector / Proxy columns identified : {len(sector_cols)}")

# Correlation with NIFTY target
corr_vals = df[sector_cols].corrwith(df[TARGET_COL]).sort_values(ascending=False)

print("\nCorrelation of Sector Features with NIFTY50 Returns:")
print("-" * 50)
for col, cv in corr_vals.head(10).items():
    bar = "█" * int(abs(cv) * 30)
    print(f"  {col:<35} {cv:>+.4f}  {bar}")
print("-" * 50)

top_sector = corr_vals.index[0]
top_corr   = corr_vals.iloc[0]
print(f"\n  Most influential feature/sector : '{top_sector}' (corr = {top_corr:+.4f})")

# Recent sector trend (last 20 days of test)
recent_n = 20
recent_target   = df[TARGET_COL].values[-recent_n:]
recent_top_sect = df[top_sector].values[-recent_n:]

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

top10 = corr_vals.head(10)
colors_bar = ['#27ae60' if v > 0 else '#e74c3c' for v in top10.values]
axes[0].barh(range(len(top10)), top10.values[::-1], color=colors_bar[::-1], edgecolor='white', alpha=0.85)
axes[0].set_yticks(range(len(top10)))
axes[0].set_yticklabels([c[:30] for c in top10.index[::-1]], fontsize=9)
axes[0].set_title('Top 10 Feature Correlations with NIFTY50 Returns', fontweight='bold')
axes[0].set_xlabel('Pearson Correlation')
axes[0].axvline(0, color='black', lw=0.8)

axes[1].plot(range(recent_n), recent_target,   label='NIFTY50 Return', color='royalblue', lw=1.8)
axes[1].plot(range(recent_n), recent_top_sect, label=f'{top_sector[:25]} (Top Driver)',
             color='darkorange', lw=1.5, ls='--')
axes[1].set_title(f'Recent Trend: NIFTY50 vs Top Driver ({top_sector[:20]})', fontweight='bold')
axes[1].set_xlabel('Days (recent 20)')
axes[1].set_ylabel('Return')
axes[1].legend(fontsize=9)

plt.suptitle('Sector Influence Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\n  Sector-Model Link:")
print(f"  The feature '{top_sector}' has the strongest correlation ({top_corr:+.4f})")
print(f"  with NIFTY50 returns. The LSTM model has learned this relationship")
print(f"  and uses this sector's momentum to generate the 5-day forecast.")
if top_corr > 0:
    print(f"  → If this sector continues its recent trend, NIFTY50 is likely to follow.")
else:
    print(f"  → This sector shows inverse relationship — watch for divergence signals.")


## 17. Final Structured Output Table

A consolidated table combining point forecast, market signal, and risk assessment for each forecast day.


In [ ]:
# ── Final Structured Output Table ───────────────────────────────────────────
BULL_THRESHOLD =  0.003
BEAR_THRESHOLD = -0.003
RISK_THRESHOLD =  0.010   # |predicted return| > 1% → high risk day

print("=" * 95)
print(" NIFTY50 — 5-DAY STRUCTURED FORECAST OUTPUT ".center(95))
print("=" * 95)
print(f"  {'Day':<4} {'Date':<14} {'Pred Return':>12} {'Pred % Chg':>11} "
      f"{'Market Signal':>14} {'Risk Level':>12}")
print("-" * 95)

for i, (bd, fv) in enumerate(zip(biz_dates, forecasts)):
    pct = fv * 100

    if fv > BULL_THRESHOLD:
        signal = "🟢 BULLISH"
    elif fv < BEAR_THRESHOLD:
        signal = "🔴 BEARISH"
    else:
        signal = "🟡 NEUTRAL"

    risk = "⚠️ HIGH RISK" if abs(fv) > RISK_THRESHOLD else "✅ LOW RISK"
    print(f"  {i+1:<4} {str(bd.date()):<14} {fv:>+12.6f} {pct:>+10.3f}%  {signal:<16} {risk}")

print("=" * 95)
print()
print("  Legend:")
print("  🟢 BULLISH  → Predicted return > +0.3% (consider long position)")
print("  🔴 BEARISH  → Predicted return < -0.3% (consider exiting / hedging)")
print("  🟡 NEUTRAL  → Small predicted move (-0.3% to +0.3%) (hold / observe)")
print("  ✅ LOW RISK → |return| ≤ 1.0% (tight prediction, lower uncertainty)")
print("  ⚠️ HIGH RISK → |return| > 1.0% (larger expected swing, trade carefully)")


## 18. Final Conclusion

### Project Summary

This notebook implements a complete **LSTM-based point forecasting pipeline** for NIFTY50 daily returns, incorporating:

1. **Systematic hyperparameter tuning** via grid search (54 combinations) — model selected on lowest validation RMSE.
2. **Stacked LSTM architecture** with BatchNormalization and Dropout for stable, generalised learning.
3. **Robust evaluation** — RMSE, MAE, R², MAPE, and Directional Accuracy on train/val/test splits.
4. **Baseline comparison** — LSTM performance benchmarked against naive persistence and zero-forecast models.
5. **Error analysis** — high-error periods linked to market volatility clusters.
6. **Sector analysis** — identification of the most influential driver feature for NIFTY50.
7. **Financial interpretation** — raw return predictions translated into Bullish/Bearish/Neutral signals with risk labels.

### Key Insight
LSTM is particularly suited for this task because financial returns are **serially dependent** and influenced by dozens of technical and sector features. The LSTM's gating mechanism captures these **long-range temporal dependencies** that traditional models (ARIMA, linear regression) cannot.

### Uncertainty-Aware Extension
Point forecasting provides a single predicted value, but markets are inherently uncertain. The companion **Quantile Forecasting Notebook** extends this work by predicting multiple quantiles (Q10–Q90), enabling probability intervals and risk-aware decision making.

### Usefulness for Financial Decision-Making
- **Fund managers**: Use directional accuracy + confidence intervals for position sizing.
- **Risk desks**: Use error analysis to identify market regimes where model reliability is lower.
- **Quant researchers**: Use RMSE improvement over baseline to validate strategy alpha.
